# Analyzing multi-dataset multi-algorithm evaluation with `bbt-test`

---

## Introduction

`bbt-test` enables researchers to compare their evaluation results using Bayesian Testing. In contrast to NHST (Null Hypothesis Significance Testing), Bayesian Testing:

### Provides actual probability of one algorithm being better than another

The Bayesian Bradley-Terry (BBT) model provides the actual probability of algorithm $A_1$ being better than algorithm $A_2$ given the observed data $D$:

$$ P(A_1 > A_2 | D) $$

while NHST reports the probability of a certain statistic $t(D)$ being larger than a critical value $\tau$, given the null hypothesis $H_0$ that one algorithm is no better than another:

$$ P(t(D) > \tau | A_1 \leq A_2) $$

### Can differentiate between "no difference" and "inconclusive"

BBT result interpretations fall into 4 categories:
- $A_1$ is better than $A_2$
- $A_2$ is better than $A_1$
- No difference between $A_1$ and $A_2$
- Inconclusive results

NHST can only differentiate between "significant difference" and "no significant difference".

### Does not rely on an arbitrary significance level

NHST relies on an arbitrary significance level $\alpha$ (e.g., 0.05) to determine whether results are statistically significant. Bayesian Testing instead relies on ROPE (Region of Practical Equivalence) to determine whether results are practically significant.

If you are interested in learning more about Bayesian Testing and the Bradley-Terry model, we recommend the following articles:

[Wainer, Jacques. "A Bayesian Bradley-Terry model to compare multiple ML algorithms on multiple data sets." Journal of Machine Learning Research 24.341 (2023): 1-34.](https://www.jmlr.org/papers/volume24/22-0907/22-0907.pdf)

[Benavoli, Alessio, et al. "Time for a change: a tutorial for comparing multiple classifiers through Bayesian analysis." Journal of Machine Learning Research 18.77 (2017): 1-36.](https://www.jmlr.org/papers/volume18/16-305/16-305.pdf)

## Practical guide

In this notebook we will show how to use `bbt-test` to analyze the results of a comparison of 25 models on 25 datasets, taken from [Benchmarking pretrained molecular embedding models for molecular representation learning](https://arxiv.org/pdf/2508.06199?), reproducing tables from the original article.

We will start by loading and preparing the data, then fit the BBT model and analyze the results, and finally compare how different ROPE values affect the interpretation.

Steps:
1. [Loading the data](#loading-the-data)
2. [Fitting the BBT model](#fitting-the-bbt-model)
3. [Analyzing the results given the ROPE value](#analyzing-the-results-given-the-rope-value)
4. [Comparing the results across different ROPE values](#comparing-the-results-across-different-rope-values)
5. [Weak vs strong interpretation](#weak-vs-strong-interpretation)

## Necessary imports

In [1]:
import pandas as pd

import bbttest

## Loading the data

`PyBBT` expects data in wide format, where each row corresponds to a dataset, columns correspond to algorithms, and values correspond to the algorithm's performance on that dataset. An additional column indicating the dataset name can be included, but it is not required when each dataset has only one run.

In [2]:
df = pd.read_csv("../datasets/benchmarking_mol.csv")
df.head()

,dataset,AtomPair_count,CDDD,CLAMP,ChemBERTa-10M-MTR,ChemFM-3B,ChemGPT-4.7M,ECFP_count,GEM,GNN-GraphCL-sum,...,chemformer_mask,coati,grover_large,mat_masking_2M,mol2vec,mol_r_tag_1024,molbert,rmat_4M,unimolv1,unimolv2
0,AMES,0.814525,0.835524,0.845998,0.842645,0.819407,0.733519,0.848170,0.712631,0.778957,...,0.801707,0.806224,0.814941,0.817682,0.806354,0.812781,0.853984,0.855419,0.772552,0.805847
1,Bioavailability_Ma,0.712005,0.668773,0.641836,0.744596,0.670436,0.695710,0.691054,0.700532,0.676921,...,0.703193,0.621716,0.883937,0.686731,0.726305,0.587463,0.683572,0.728301,0.660126,0.765381
2,CYP1A2_Veith,0.928864,0.923714,0.950784,0.926163,0.903825,0.867991,0.927867,0.875383,0.897117,...,0.904443,0.897014,0.856961,0.926905,0.913344,0.901323,0.929359,0.930857,0.901899,0.915601
3,CYP2C19_Veith,0.858162,0.872114,0.922305,0.871285,0.853653,0.789982,0.874444,0.770266,0.834949,...,0.839680,0.848842,0.779436,0.874605,0.851539,0.851361,0.885187,0.880757,0.826559,0.863196
4,CYP2C9_Substrate_CarbonMangels,0.596310,0.655182,0.697775,0.692078,0.667933,0.652469,0.627238,0.597396,0.658709,...,0.640532,0.672409,0.572165,0.624525,0.644872,0.604992,0.646636,0.666305,0.594954,0.679327


## Fitting the BBT model

With the data loaded, we can set up the BBT model and fit the posterior distribution. Once fitted, the model object stores the sampling results, allowing multiple analyses to be run without re-fitting.

We set `local_rope_value` to $0.01$, meaning that a difference of $0.01$ or less in the reported metric is considered practically negligible. If multiple runs per dataset are provided, PyBBT will calculate this threshold automatically using in-dataset variance.

We use `spread` as the `tie_solver` (the default). The authors of the BBT model recommend any of `spread`, `add`, or `forget`, noting similar performance across all three. We also use the default prior distribution — `log_normal` with scale $1.0$, corresponding to $\sigma \sim \text{LogNormal}(0, 1)$.

Finally, we set `maximize=True` to indicate that higher values of the reported metric are better.

In [3]:
model = bbttest.PyBBT(
    local_rope_value=0.01,
    hyper_prior="log_normal",
    tie_solver="spread",
    maximize=True,
    scale=1.0,
)

Next, we fit the model via MCMC sampling to obtain the posterior distribution. **Remember** — if the dataframe contains a dataset column, it must be specified via `dataset_col` in `fit()`. Otherwise, PyBBT will treat it as just another algorithm, leading to errors or incorrect results.

By default, PyBBT uses 4 chains with 1000 draws each (PyMC defaults).

In [4]:
model.fit(
    data=df,
    dataset_col="dataset",
    random_seed=42,
)

Constructing win table:   0%|          | 0/300 [00:00<?, ?it/s]

Multiprocess sampling (4 chains in 4 jobs)
CompoundStep
>NUTS: [sigma, beta]
>Metropolis: [win1_rep]


Output()

Sampling 4 chains for 1_000 tune and 1_000 draw iterations (4_000 + 4_000 draws total) took 13 seconds.
The rhat statistic is larger than 1.01 for some parameters. This indicates problems during sampling. See https://arxiv.org/abs/1903.08008 for details
The effective sample size per chain is smaller than 100 for some parameters.  A higher number is needed for reliable rhat and ess computation. See https://arxiv.org/abs/1903.08008 for details


Once the model is fitted, we can run multiple analyses on the results.

## Analyzing the results given the ROPE value

We will perform the analysis using the same ROPE value as in the original article ($35\% - 65\%$).

In [5]:
model.posterior_table(rope_value=(0.35, 0.65))

,pair,mean,delta,above_50,in_rope,weak_interpretation
0,CLAMP > rmat_4M,0.55,0.11,0.91,1.00,Equivalent
1,CLAMP > molbert,0.60,0.11,1.00,0.95,CLAMP better
2,CLAMP > CDDD,0.61,0.10,1.00,0.87,CLAMP better
3,CLAMP > ChemBERTa-10M-MTR,0.62,0.11,1.00,0.84,CLAMP better
4,CLAMP > ECFP_count,0.62,0.11,1.00,0.83,CLAMP better
...,...,...,...,...,...,...
295,ChemGPT-4.7M > SELFormer-Lite,0.56,0.12,0.94,0.99,Equivalent
296,ChemGPT-4.7M > GraphFP-CP,0.87,0.07,1.00,0.00,ChemGPT-4.7M better
297,SimSon > SELFormer-Lite,0.56,0.12,0.93,1.00,Equivalent
298,SimSon > GraphFP-CP,0.87,0.07,1.00,0.00,SimSon better


In this case, the result table contains 300 rows, corresponding to all pairwise comparisons. To make it more readable, we can filter the results to include only comparisons against a specific model (called control).

In [6]:
model.posterior_table(
    rope_value=(0.35, 0.65),
    control_model="ECFP_count",
)

,pair,mean,delta,above_50,in_rope,weak_interpretation
0,CLAMP > ECFP_count,0.62,0.11,1.00,0.83,CLAMP better
1,rmat_4M > ECFP_count,0.57,0.11,0.98,0.99,Equivalent
2,molbert > ECFP_count,0.52,0.11,0.74,1.00,Equivalent
3,CDDD > ECFP_count,0.50,0.11,0.55,1.00,Equivalent
4,ChemBERTa-10M-MTR > ECFP_count,0.50,0.10,0.52,1.00,Equivalent
5,ECFP_count > mat_masking_2M,0.51,0.11,0.63,1.00,Equivalent
6,ECFP_count > AtomPair_count,0.55,0.11,0.94,1.00,Equivalent
7,ECFP_count > MoLFormer-XL-both-10pct,0.60,0.10,1.00,0.93,ECFP_count better
8,ECFP_count > mol2vec,0.64,0.10,1.00,0.66,ECFP_count better
9,ECFP_count > TT,0.70,0.09,1.00,0.03,ECFP_count better


We can also filter the results to include only specific list of models:

In [7]:
model.posterior_table(
    rope_value=(0.35, 0.65),
    control_model="ECFP_count",
    selected_models=["CLAMP", "TT", "ChemFM-3B"],
)

,pair,mean,delta,above_50,in_rope,weak_interpretation
0,CLAMP > ECFP_count,0.62,0.11,1.0,0.83,CLAMP better
1,ECFP_count > TT,0.70,0.09,1.0,0.03,ECFP_count better
2,ECFP_count > ChemFM-3B,0.74,0.08,1.0,0.00,ECFP_count better


## Comparing the results across different ROPE values

While the BBT model does not depend on a single fixed ROPE value, the choice of ROPE does influence the interpretation. To assess the sensitivity of the results to this choice, we can run the analysis across multiple values. Keep in mind that a larger ROPE requires a greater performance difference before declaring one model superior.

We compare BBT interpretations against the control model (`ECFP_count`) for the following ROPE values:
- $45\% - 55\%$ (default recommended by the BBT authors)
- $40\% - 60\%$
- $35\% - 65\%$ (used in the original article)

And a more extreme value:
- $20\% - 80\%$

In [8]:
model.rope_comparison_control_table(
    rope_values=((0.45, 0.55), (0.40, 0.60), (0.35, 0.65), (0.20, 0.80)),
    control_model="ECFP_count",
)

,rope_value,better_models,equivalent_models,worse_models,unknown_models
0,"(0.45, 0.55)","CLAMP, rmat_4M",,"MoLFormer-XL-both-10pct, mol2vec, TT, ChemFM-3...","molbert, CDDD, ChemBERTa-10M-MTR, mat_masking_..."
1,"(0.4, 0.6)","CLAMP, rmat_4M","molbert, CDDD, ChemBERTa-10M-MTR, mat_masking_2M","MoLFormer-XL-both-10pct, mol2vec, TT, ChemFM-3...",AtomPair_count
2,"(0.35, 0.65)",CLAMP,"rmat_4M, molbert, CDDD, ChemBERTa-10M-MTR, mat...","MoLFormer-XL-both-10pct, mol2vec, TT, ChemFM-3...",
3,"(0.2, 0.8)",,"CLAMP, rmat_4M, molbert, CDDD, ChemBERTa-10M-M...","mol_r_tag_1024, coati, GraphMVP_CP-max, chemfo...",


## Weak vs strong interpretation

There are two ways to interpret the posterior distribution of the BBT model. As the authors suggest:

> We believe that there are two main approaches to utilizing the BBT model in research. For researchers or audiences who are more familiar with the frequentist approach, we recommend using the summary values related to the weak interpretation of the parameters (the `above.50` and `in.rope` values) and a $0.95$ probability threshold. This approach utilizes familiar threshold numbers such as $0.95$ or $95\%$.

> If researchers and their audiences are more comfortable with Bayesian results, we recommend following the strong interpretation (and the mean summary of the probabilities) and choosing a threshold of $0.70$.

> — Wainer (2023)

Note: the paper uses R-style column names (`above.50`, `in.rope`); in `bbt-test` these are `above_50` and `in_rope`.

Before exploring the two interpretations, let's revisit all the columns returned by `posterior_table`. By default it returns only the subset needed for weak interpretation. Let's look at the full table:

In [11]:
model.posterior_table(
    rope_value=(0.35, 0.65),
    control_model="ECFP_count",
    selected_models=["CLAMP", "TT", "ChemFM-3B"],
    columns=bbttest.PyBBT.ALL_PROPERTIES_COLUMNS,
)

,pair,left_model,right_model,median,mean,hdi_low,hdi_high,delta,above_50,in_rope,weak_interpretation,strong_interpretation
0,CLAMP > ECFP_count,CLAMP,ECFP_count,0.62,0.62,0.56,0.67,0.11,1.0,0.83,CLAMP better,Unknown
1,ECFP_count > TT,ECFP_count,TT,0.70,0.70,0.66,0.74,0.09,1.0,0.03,ECFP_count better,ECFP_count better
2,ECFP_count > ChemFM-3B,ECFP_count,ChemFM-3B,0.74,0.74,0.70,0.78,0.08,1.0,0.00,ECFP_count better,ECFP_count better


The key columns describing the posterior distribution for each model pair are:
- `mean`: the average probability of `left_model` being better than `right_model`.
- `median`: the median probability of `left_model` being better than `right_model`.
- `hdi_low` and `hdi_high`: the lower and upper bounds of the Highest Density Interval (HDI) of the posterior.
- `above_50`: proportion of posterior samples where $P(\text{left\_model} > \text{right\_model}) > 0.5$.
- `in_rope`: proportion of posterior samples where $P(\text{left\_model} > \text{right\_model})$ falls within the ROPE.

**Weak interpretation** uses `above_50` and `in_rope` as follows:
1. If `in_rope` $\geq 0.95$ → models are **equivalent**.
2. Else if `above_50` $\geq 0.95$ → `left_model` is **better**.
3. Otherwise → result is **inconclusive** (Unknown).

Model pairs are always ordered by the internal posterior $\beta$ estimate of global performance, so `left_model` is always the globally stronger model in the pair.

**Strong interpretation** uses `mean`:
1. If `mean` $> 0.70$ → `left_model` is **better**.
2. If `mean` $\leq 0.55$ → models are **equivalent**.
3. If $0.55 <$ `mean` $\leq 0.70$ → result is **inconclusive** (Unknown).